# Machine Learning Zoom Camp Decision Tree Regressor

In [17]:
import numpy as np
import pandas as pd

In [18]:
# Q1
np.__version__

'2.0.1'

In [19]:

df = pd.read_csv("car_fuel_efficiency_2026.csv")

In [20]:
missing = df.isnull().sum()
print(missing[missing > 0])

horsepower      877
acceleration    264
dtype: int64


In [21]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer

# Fill missing values with zeros
df = df.fillna(0)

# Split dataset
df_full_train, df_test = train_test_split(
    df, test_size=0.2, random_state=1
)

df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=1
)

# Separate target variable
y_train = df_train.fuel_efficiency_mpg.values
y_val = df_val.fuel_efficiency_mpg.values
y_test = df_test.fuel_efficiency_mpg.values

# Drop target variable from feature datasets
del df_train['fuel_efficiency_mpg']
del df_val['fuel_efficiency_mpg']
del df_test['fuel_efficiency_mpg']

# Convert DataFrames to dictionaries
train_dict = df_train.to_dict(orient='records')
val_dict = df_val.to_dict(orient='records')
test_dict = df_test.to_dict(orient='records')

# Initialize DictVectorizer
dv = DictVectorizer(sparse=True)

# Fit and transform training data
X_train = dv.fit_transform(train_dict)

# Transform validation and test data
X_val = dv.transform(val_dict)
X_test = dv.transform(test_dict)

In [22]:

from sklearn.tree import DecisionTreeRegressor

# Train decision tree regressor
dt = DecisionTreeRegressor(max_depth=1)
dt.fit(X_train, y_train)

# Get the feature used for splitting
feature_index = dt.tree_.feature[0]

# Get the DictVectorizer feature name
feature_name = dv.get_feature_names_out()[feature_index]

print("Splitting feature:", feature_name)

Splitting feature: model_year


In [23]:

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import numpy as np

# Train Random Forest Regressor
rf = RandomForestRegressor(
    n_estimators=10,
    random_state=1,
    n_jobs=-1
)

rf.fit(X_train, y_train)

# Make predictions on validation data
y_pred = rf.predict(X_val)

# Calculate RMSE
rmse = np.sqrt(mean_squared_error(y_val, y_pred))

print("Validation RMSE:", rmse)


Validation RMSE: 1.837054054730018


In [24]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import numpy as np

estimators = [10, 50, 100, 150]

for n in estimators:
    rf = RandomForestRegressor(
        n_estimators=n,
        random_state=1,
        n_jobs=-1
    )

    rf.fit(X_train, y_train)

    y_pred = rf.predict(X_val)

    rmse = np.sqrt(mean_squared_error(y_val, y_pred))

    print(f"n_estimators: {n}, RMSE: {rmse:.3f}")


results = {}

for n in [10, 50, 100, 150]:
    rf = RandomForestRegressor(
        n_estimators=n,
        random_state=1,
        n_jobs=-1
    )

    rf.fit(X_train, y_train)

    y_pred = rf.predict(X_val)

    rmse = np.sqrt(mean_squared_error(y_val, y_pred))

    results[n] = round(rmse, 3)

best_n = min(results, key=results.get)

print("RMSE results:", results)
print("Best n_estimators:", best_n)
print("Lowest validation RMSE:", results[best_n])


n_estimators: 10, RMSE: 1.837
n_estimators: 50, RMSE: 1.772
n_estimators: 100, RMSE: 1.768
n_estimators: 150, RMSE: 1.769
RMSE results: {10: np.float64(1.837), 50: np.float64(1.772), 100: np.float64(1.768), 150: np.float64(1.769)}
Best n_estimators: 100
Lowest validation RMSE: 1.768


In [25]:

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import numpy as np

max_depth_values = [10, 15, 20, 25]
n_estimators_values = [10, 50, 100, 150]

scores = {}

for depth in max_depth_values:
    rmse_values = []

    for n in n_estimators_values:
        rf = RandomForestRegressor(
            n_estimators=n,
            max_depth=depth,
            random_state=1,
            n_jobs=-1
        )

        rf.fit(X_train, y_train)

        y_pred = rf.predict(X_val)

        rmse = np.sqrt(mean_squared_error(y_val, y_pred))

        rmse_values.append(rmse)

    mean_rmse = np.mean(rmse_values)
    scores[depth] = round(mean_rmse, 3)

    print(f"max_depth: {depth}, mean RMSE: {mean_rmse:.3f}")

# Find best max_depth
best_depth = min(scores, key=scores.get)

print("\nBest max_depth:", best_depth)
print("Lowest mean RMSE:", scores[best_depth])


max_depth: 10, mean RMSE: 1.755
max_depth: 15, mean RMSE: 1.784
max_depth: 20, mean RMSE: 1.788
max_depth: 25, mean RMSE: 1.786

Best max_depth: 10
Lowest mean RMSE: 1.755


In [26]:

from sklearn.ensemble import RandomForestRegressor
import pandas as pd

# Train Random Forest model
rf = RandomForestRegressor(
    n_estimators=10,
    max_depth=20,
    random_state=1,
    n_jobs=-1
)

rf.fit(X_train, y_train)

# Extract feature importances
feature_importances = rf.feature_importances_

# Create a DataFrame
importance_df = pd.DataFrame({
    'feature': dv.get_feature_names_out(),
    'importance': feature_importances
})

# Sort by importance
importance_df = importance_df.sort_values(
    by='importance',
    ascending=False
)

print(importance_df)

# Most important feature
print("\nMost important feature:")
print(importance_df.iloc[0]['feature'])


                         feature  importance
9                     model_year    0.296001
15                vehicle_weight    0.203879
6             fuel_type=Gasoline    0.160712
8                     horsepower    0.078017
4            engine_displacement    0.061005
0                   acceleration    0.055201
14                    origin=USA    0.042144
2   drivetrain=Front-wheel drive    0.020145
11                     num_doors    0.019415
5               fuel_type=Diesel    0.017038
7               fuel_type=Hybrid    0.015800
1     drivetrain=All-wheel drive    0.007682
10                 num_cylinders    0.007284
12                   origin=Asia    0.006391
3    drivetrain=Rear-wheel drive    0.005781
13                 origin=Europe    0.003505

Most important feature:
model_year


In [27]:
print(importance_df.head(4))

               feature  importance
9           model_year    0.296001
15      vehicle_weight    0.203879
6   fuel_type=Gasoline    0.160712
8           horsepower    0.078017


In [28]:
importance_df[
    importance_df['feature'].isin([
        'vehicle_weight',
        'horsepower',
        'acceleration',
        'engine_displacement'
    ])
].sort_values('importance', ascending=False)

,feature,importance
15,vehicle_weight,0.203879
8,horsepower,0.078017
4,engine_displacement,0.061005
0,acceleration,0.055201


In [29]:
%pip install xgboost

Note: you may need to restart the kernel to use updated packages.


In [30]:

import xgboost as xgb
import numpy as np
from sklearn.metrics import mean_squared_error

# Create DMatrix
dtrain = xgb.DMatrix(X_train, label=y_train)
dval = xgb.DMatrix(X_val, label=y_val)

# Create watchlist
watchlist = [
    (dtrain, 'train'),
    (dval, 'val')
]

# Base parameters
xgb_params = {
    'eta': 0.3,
    'max_depth': 6,
    'min_child_weight': 1,
    'objective': 'reg:squarederror',
    'nthread': 8,
    'seed': 1,
    'verbosity': 1
}

# Train with eta = 0.3
model_03 = xgb.train(
    xgb_params,
    dtrain,
    num_boost_round=100,
    evals=watchlist
)

# Calculate validation RMSE
y_pred_03 = model_03.predict(dval)
rmse_03 = np.sqrt(mean_squared_error(y_val, y_pred_03))

print("eta = 0.3, RMSE:", round(rmse_03, 3))


# Change eta to 0.1
xgb_params['eta'] = 0.1

# Train with eta = 0.1
model_01 = xgb.train(
    xgb_params,
    dtrain,
    num_boost_round=100,
    evals=watchlist
)

# Calculate validation RMSE
y_pred_01 = model_01.predict(dval)
rmse_01 = np.sqrt(mean_squared_error(y_val, y_pred_01))

print("eta = 0.1, RMSE:", round(rmse_01, 3))


# Compare results
if rmse_03 < rmse_01:
    print("Best eta: 0.3")
else:
    print("Best eta: 0.1")


[0]	train-rmse:2.40653	val-rmse:2.46782
[1]	train-rmse:2.07098	val-rmse:2.15784
[2]	train-rmse:1.86454	val-rmse:1.97398
[3]	train-rmse:1.73360	val-rmse:1.86662
[4]	train-rmse:1.65059	val-rmse:1.80312


[5]	train-rmse:1.59328	val-rmse:1.76216
[6]	train-rmse:1.55435	val-rmse:1.74392
[7]	train-rmse:1.52677	val-rmse:1.73976
[8]	train-rmse:1.50265	val-rmse:1.73478
[9]	train-rmse:1.48761	val-rmse:1.73282
[10]	train-rmse:1.47125	val-rmse:1.73205
[11]	train-rmse:1.45902	val-rmse:1.73227
[12]	train-rmse:1.44829	val-rmse:1.73503
[13]	train-rmse:1.43604	val-rmse:1.73466
[14]	train-rmse:1.42972	val-rmse:1.73448
[15]	train-rmse:1.41908	val-rmse:1.73512
[16]	train-rmse:1.41206	val-rmse:1.73297
[17]	train-rmse:1.40940	val-rmse:1.73517
[18]	train-rmse:1.40278	val-rmse:1.73584
[19]	train-rmse:1.39637	val-rmse:1.73664
[20]	train-rmse:1.39368	val-rmse:1.73766
[21]	train-rmse:1.39254	val-rmse:1.73701
[22]	train-rmse:1.37639	val-rmse:1.74216
[23]	train-rmse:1.36470	val-rmse:1.74363
[24]	train-rmse:1.36071	val-rmse:1.74388
[25]	train-rmse:1.35837	val-rmse:1.74359
[26]	train-rmse:1.35382	val-rmse:1.74430
[27]	train-rmse:1.34456	val-rmse:1.74461
[28]	train-rmse:1.34092	val-rmse:1.74330
[29]	train-rmse:1.339